Encode polygons using quad tree, uniform grid, and filtered uniform for Parks 50k.
Weighted JD with int and real number-based encoding.
Use cus-nmslib environment which has the latest nmslib update.

In [2]:
# import library
import numpy as np
# import geopandas
from shapely.geometry.polygon import Polygon
from multiprocessing import Process, Array, Pool
from shapely.strtree import STRtree
from shapely import affinity
import matplotlib.pyplot as plt
import numpy as np
import shapely.wkt
import math
import numpy 
import sys 
import nmslib 
import time 
import timeit
from scipy.sparse import csr_matrix 
from sklearn.model_selection import train_test_split 
print(sys.version)
print("NMSLIB version:", nmslib.__version__)

import os

from multiprocessing import Process, Array

# to set the path
import sys
sys.path.insert(1, '../lib/')
import wkthelper
from quadtree import quadtree
from grid import grid

3.9.20 (main, Oct  3 2024, 07:27:41) 
[GCC 11.2.0]
NMSLIB version: VERSION_INFO


Read polygon feature vector files

In [3]:
# sort files in a folder by id value
def sortFilesById(files):
    ids=[]
    for file in files:
        ids.append(int(file[file.find('_')+1: file.find('.txt')]))
    
    ids, filenames = zip(*sorted(zip(ids, files)))
    return filenames

# read sparse vector data to int strings
def readAllSparseStr(path, max_qty = None): 
    # read all files in the folder and sort them
    files=os.listdir(path)
    files=sortFilesById(files)
    
    vector_str=[]
    for file in files:
        vector_str+=readEncodeFile(path+file)
    return vector_str

# read a encoding file into a 2-D array
def readEncodeFileToArray(file):
    # print(file)
    # f=open(file, 'r')
    # data=f.readlines()
    # f.close()
    data = np.loadtxt(file, dtype=float).tolist()

    return data

# read all files in a directory to a 2-D array
def readAllSparseArray(path, max_qty = None): 
    # read all files in the folder and sort them
    files=os.listdir(path)
    files=sortFilesById(files)
    
    vectors=[]
    for file in files:
        vectors+=(readEncodeFileToArray(path+file))
    return vectors


Indexing with NMSLIB

In [4]:
# creating index using vectors in string data *************
def createWeightedJaccardIndex(data_vector_str, M, efC, num_threads, post):
    index_time_params = {'M': M, 'indexThreadQty': num_threads, 'efConstruction': efC, 'post' : post}

    # Intitialize the library, specify the space, the type of the data and add data points 
    # Note that in the GENERIC case, data points are passed as strings!

    # index = nmslib.init(method='hnsw', space='bit_jaccard', data_type=nmslib.DataType.OBJECT_AS_STRING) 
    index = nmslib.init(method='hnsw', space='WeightedJaccard') 
    index.addDataPointBatch(data_vector_str) 

    # Create an index
    start = time.time()
    index.createIndex(index_time_params) 
    end = time.time() 
    print("Index-time parameters [{}] Indexing time: {}".format(index_time_params, (end-start)))
    print("{} vectors indexed".format(len(data_vector_str)))

    return index

# creating index using vectors in string data *************
def createBitJaccardIndex(data_vector_str, M, efC, num_threads, post):
    index_time_params = {'M': M, 'indexThreadQty': num_threads, 'efConstruction': efC, 'post' : post}

    # Intitialize the library, specify the space, the type of the data and add data points 
    # Note that in the GENERIC case, data points are passed as strings!

    index = nmslib.init(method='hnsw', space='jaccard_sparse', data_type=nmslib.DataType.OBJECT_AS_STRING) 
    index.addDataPointBatch(data_vector_str) 

    # Create an index
    start = time.time()
    index.createIndex(index_time_params) 
    end = time.time() 
    print("Index-time parameters [{}] Indexing time: {}".format(index_time_params, (end-start)))
    print("{} vectors indexed".format(len(data_vector_str)))

    return index

Querying

In [5]:
# *******
def query(index, query_vector_str, K, efS, num_threads):
      # Setting query-time parameters
      query_time_params = {'efSearch': efS}
      print('Setting query-time parameters', query_time_params)
      index.setQueryTimeParams(query_time_params) 

      # Querying
      query_qty = len(query_vector_str)
      start = time.time() 
      nbrs = index.knnQueryBatch(query_vector_str, k = K, num_threads = num_threads)
      end = time.time() 
      print('kNN time total=%f (sec), per query=%f (sec), per query adjusted for thread number=%f (sec)' % 
            (end-start, float(end-start)/query_qty, num_threads*float(end-start)/query_qty)) 
      return nbrs

Groud truth file reading and compute recall

In [6]:
# More convinient and latest method to read ground truth files without bach concept ************
# read a ground truth file to list
def readGroundTruthFile(filename):
    f=open(filename, 'r')
    lines=f.readlines()
    f.close()

    gt=[]
    for line in lines:
        dataRow=line.split(', ')
        if len(dataRow)<=1:
            gt.append([])
        else:
            gtRow=[]
            for data in dataRow:
                gtRow.append(int(data))
            gt.append(gtRow)
    return gt

# sort files in a folder by id value
def sortFilesByIdGT(files):
    ids=[]
    for file in files:
        ids.append(int(file[file.find('_')+1: file.find('-')]))
    
    ids, filenames = zip(*sorted(zip(ids, files)))
    return filenames
# filter out k similar values for each query polygon
def selectKGroundTruth(gs_all, k):
    gs=[]
    for i in range(len(gs_all)):
        gt=[]
        j=0
        while(j < len(gs_all[i]) and k > j):
            gt.append(gs_all[i][j])
            j+=1
        gs.append(gt)
    return gs

# compute recall when ground truth only has data for query data  
def computeRecallQueryOnly(gs, nbrs, query_qty):
    recall=0.0
    gs_nonzero_count=0

    for i in range(0, query_qty):
        if len(gs[i])>0:
            correct_set = set(gs[i])
            ret_set = set(nbrs[i][0])
            recall = recall + float(len(correct_set.intersection(ret_set))) / len(correct_set)
            gs_nonzero_count+=1
    recall_full = recall / query_qty
    recall = recall / gs_nonzero_count
    print('Full kNN recall {}\nkNN recall {}'.format(recall_full, recall))

# read all files in the directory
def readAllGroundTruthFiles(path):
    files=os.listdir(path)
    files=sortFilesByIdGT(files)
    print(len(files))

    gtArray=[]
    for file in files:
        gtArray=gtArray+readGroundTruthFile(path+file)
    
    return gtArray

In [7]:
# break
if

SyntaxError: invalid syntax (1755648541.py, line 2)

In [ ]:
# testing autoencorder-based encoding
# read file data
# 10k dataset 135x135x4

all_vector=readAllSparseArray("/raid/ssEncodingData/encoding/auto-pk-135_4-l12-query-8000/")   # path to encoding files
filename="/raid/ssEncodingData/warehouse/pk-query-50k/"   # path to ground truth folder


start=0
end=len(all_vector)
data_percet=0.8
data_start=start
data_end=math.ceil((end-start)*data_percet)
query_start=data_end
query_end=end

data_vector=all_vector[data_start:data_end]
print("{} vectors found in data!".format(len(data_vector)))
query_vector=all_vector[query_start:query_end]
print("{} vectors found in query!".format(len(query_vector)))

# Index parameters
# ------------------------------------------------------------------
# How many neighbors for a given node at non-zero layers (5-100)
# At non-zero layer (bottom most) ther are 2*M neighbors (maxM0 variable to change default)
M = 20
# maxM0 =2**20

# efC = 500            #ef construction (100-2000)
efC = 200            #ef construction (100-2000)
num_threads = 50         # numbers of threads used for training
post=1               #will do extra processing with [1, 2] values. No extra processing with 0

# Setting query-time parameters
efS = 200
# ------------------------------------------------------------------

# create index
index=createWeightedJaccardIndex(data_vector, M, efC, num_threads, post)

# time 26m 44s

# Using query data only ground truth
# This system does not use a batch size

# ------------------------------------------------------------------

# Query and recall computation
K=500      #top k neighbors are returned in the results
# Query
nbrs=query(index, query_vector, K, efS, num_threads)

gs=readAllGroundTruthFiles(filename)
# select k ground truth
gsk=selectKGroundTruth(gs, K)
# Calculate recall rate
computeRecallQueryOnly(gsk, nbrs, query_qty=len(nbrs))


8000 vectors found in data!
2000 vectors found in query!
Index-time parameters [{'M': 20, 'indexThreadQty': 50, 'efConstruction': 200, 'post': 1}] Indexing time: 606.0682635307312
8000 vectors indexed
Setting query-time parameters {'efSearch': 200}
kNN time total=29.334513 (sec), per query=0.014667 (sec), per query adjusted for thread number=0.733363 (sec)
60
Full kNN recall 0.32365160523971465
kNN recall 0.3560523710007862
